# Financial Crime Risk Analytics — Review Walkthrough

This notebook is a reviewer-facing companion to the scalable Elliptic2 pipeline. It focuses on the evidence behind model selection and investigator-review prioritization.

The project is **decision support only**. A model score prioritizes human review; it does not establish criminal activity, make a legal determination, or automate regulatory reporting.


## 1. Establish the evaluation frame

With a roughly 2.27% suspicious prevalence, accuracy would hide the problem. The project therefore emphasizes PR-AUC globally and precision/recall/lift inside constrained review budgets.


In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
findings = (ROOT / "reports" / "CURRENT_FINDINGS.md").read_text()

# Start with the evaluation frame because rare-event modeling needs different success metrics than balanced classification.
evaluation_frame = {
    "labeled components": 121_810,
    "licit components": 119_047,
    "suspicious components": 2_763,
    "positive prevalence": "about 2.27%",
    "primary global metric": "PR-AUC",
    "primary operational lens": "investigator review budget",
}
evaluation_frame


{'labeled components': 121810, 'licit components': 119047, 'suspicious components': 2763, 'positive prevalence': 'about 2.27%', 'primary global metric': 'PR-AUC', 'primary operational lens': 'investigator review budget'}

### What this tells us

The project is evaluating a rare-event ranking problem, not a balanced classification exercise. PR-AUC measures how well the model concentrates suspicious components overall, while review-budget metrics answer the operational question: **if investigators can examine only a small fraction of cases, how useful is the ordering?**


## 2. Compare the structural baseline with the validated node-enriched model

The structural-only model is intentionally retained as the benchmark to beat. The node-enriched experiment adds information from the 43 anonymized background-node features and validates the result over repeated stratified splits.


In [2]:
# Keep the weak baseline beside the winning model so the improvement remains visible and auditable.
model_comparison = [
    {
        "model": "structural logistic regression",
        "PR-AUC": 0.0263,
        "ROC-AUC": 0.5460,
        "top-0.5% precision": "4.10%",
        "top-0.5% lift": "1.81×",
    },
    {
        "model": "node-enriched random forest",
        "PR-AUC": 0.5279,
        "ROC-AUC": 0.9278,
        "top-0.5% precision": "94.26%",
        "top-0.5% lift": "41.53×",
    },
]
model_comparison


[{'model': 'structural logistic regression', 'PR-AUC': 0.0263, 'ROC-AUC': 0.546, 'top-0.5% precision': '4.10%', 'top-0.5% lift': '1.81×'}, {'model': 'node-enriched random forest', 'PR-AUC': 0.5279, 'ROC-AUC': 0.9278, 'top-0.5% precision': '94.26%', 'top-0.5% lift': '41.53×'}]

### What this tells us

Graph structure by itself provides little useful separation. The node-enriched random forest changes the operational picture: across repeated splits it reaches about **0.528 mean PR-AUC**, and at the tightest 0.5% review budget it averages **94.26% precision and 41.53× lift over random review**.

The important result is not simply that a random forest scores well. It is that the model concentrates substantially more relevant cases inside a realistic investigator capacity constraint, and that improvement remains stable across repeated splits.


## 3. Treat the edge experiment as a real negative result

More features and more engineering do not automatically create more investigator value. The project keeps the edge-feature ablation because it changes the model-selection decision.


In [3]:
# Preserve the ablation because a negative incremental-value result is still a model-selection result.
edge_ablation = [
    {"measure": "mean PR-AUC", "node-only RF": 0.5279, "node+edge RF": 0.5022},
    {"measure": "PR-AUC SD", "node-only RF": 0.0081, "node+edge RF": 0.0171},
    {"measure": "mean ROC-AUC", "node-only RF": 0.9278, "node+edge RF": 0.9247},
    {"measure": "top-0.5% precision", "node-only RF": "94.26%", "node+edge RF": "94.10%"},
]
edge_ablation


[{'measure': 'mean PR-AUC', 'node-only RF': 0.5279, 'node+edge RF': 0.5022}, {'measure': 'PR-AUC SD', 'node-only RF': 0.0081, 'node+edge RF': 0.0171}, {'measure': 'mean ROC-AUC', 'node-only RF': 0.9278, 'node+edge RF': 0.9247}, {'measure': 'top-0.5% precision', 'node-only RF': '94.26%', 'node+edge RF': '94.10%'}]

### What this tells us

The additional 95 edge features were successfully joined and aggregated, but the resulting model is **worse and less stable**. Mean PR-AUC drops by roughly 4.9% relative to the node-only model and performance is weaker at every larger tested review budget.

That is a useful engineering conclusion: the project rejects extra complexity when it does not improve the decision objective.


## 4. Separate ranking from probability interpretation

Calibration is evaluated on a held-out 60/20/20 train/calibration/test design. The operational queue still uses the raw random-forest ranking because ranking quality is the primary requirement.


In [4]:
calibration = [
    {"method": "raw RF", "PR-AUC": 0.507435, "ECE": 0.008812, "top-122 captured": 113},
    {"method": "sigmoid", "PR-AUC": 0.507435, "ECE": 0.002716, "top-122 captured": 113},
    {"method": "isotonic", "PR-AUC": 0.481282, "ECE": 0.001389, "top-122 captured": 109},
]
calibration


[{'method': 'raw RF', 'PR-AUC': 0.507435, 'ECE': 0.008812, 'top-122 captured': 113}, {'method': 'sigmoid', 'PR-AUC': 0.507435, 'ECE': 0.002716, 'top-122 captured': 113}, {'method': 'isotonic', 'PR-AUC': 0.481282, 'ECE': 0.001389, 'top-122 captured': 109}]

### What this tells us

Sigmoid calibration improves probability-like reliability substantially without changing ranking performance, while isotonic calibration improves calibration metrics further but damages PR-AUC and captures fewer suspicious components at the tightest review budget.

For that reason, the project keeps **two semantics**: the raw RF score is the investigator-priority ranking signal, while sigmoid calibration is optional when a probability-like research estimate is useful. Neither should be interpreted as proof of criminal activity.


## 5. Make the decision boundary explicit


In [5]:
guardrails = {
    "score means": "priority for human review",
    "score does not mean": "criminal guilt or a literal suspicious-activity probability",
    "final model": "node-enriched random forest",
    "queue design": "capacity-based priority tiers",
    "human role": "investigator judgment remains required",
}
guardrails


{'score means': 'priority for human review', 'score does not mean': 'criminal guilt or a literal suspicious-activity probability', 'final model': 'node-enriched random forest', 'queue design': 'capacity-based priority tiers', 'human role': 'investigator judgment remains required'}

### What this tells us

The model is useful because it changes **where limited investigative attention starts**, not because it makes a legal or regulatory determination. Keeping that distinction visible in the notebook makes the project both more credible and safer to interpret.
